# Reading the XCAP OHLCV dataset

First look at the equity price dataset built by `xcap phase1-build`. Covers how
to load it, the two joins that matter, and the traps that will silently corrupt
a backtest if you get them wrong.

**Read [`docs/DATA_RETRIEVAL.md`](../docs/DATA_RETRIEVAL.md) for why the dataset
is shaped this way.** The short version:

- `eod/` holds **raw** OHLCV — adjusted for neither splits nor dividends.
- `adjustments/` holds factors computed locally. `adjusted = close * price_factor`.
- `vendor_adjusted_close` exists **only** for reconciliation. Do not trade on it.

In [1]:
import duckdb, pandas as pd, numpy as np
from pathlib import Path

ROOT    = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
PARQUET = ROOT / "data" / "parquet"

con = duckdb.connect()
con.execute(f"SET temp_directory='{ROOT / 'data' / '_duckdb_tmp'}'")
for name, src in {
    "eod":        f"{PARQUET}/eod/**/*.parquet",
    "adj":        f"{PARQUET}/adjustments/**/*.parquet",
    "splits":     f"{PARQUET}/splits.parquet",
    "securities": f"{PARQUET}/securities.parquet",
}.items():
    con.execute(f"CREATE VIEW {name} AS SELECT * FROM read_parquet('{src}')")

print(con.execute("DESCRIBE eod").df().to_string(index=False))

          column_name column_type null  key default extra
          security_id     INTEGER  YES None    None  None
           api_ticker     VARCHAR  YES None    None  None
                 date        DATE  YES None    None  None
                 open      DOUBLE  YES None    None  None
                 high      DOUBLE  YES None    None  None
                  low      DOUBLE  YES None    None  None
                close      DOUBLE  YES None    None  None
vendor_adjusted_close      DOUBLE  YES None    None  None
               volume      BIGINT  YES None    None  None
                 year      BIGINT  YES None    None  None


## 1. Shape of the dataset

In [2]:
con.execute('''
    SELECT COUNT(*) AS bars, COUNT(DISTINCT security_id) AS securities,
           MIN(date) AS first_date, MAX(date) AS last_date
    FROM eod
''').df()

,bars,securities,first_date,last_date
0,58985881,31513,2000-01-03,2026-07-27


In [3]:
# Bars per year. The 2000 floor is deliberate: the vendor's delisted archive
# begins ~1997-98, so no earlier start year is survivorship-bias free.
by_year = con.execute('''
    SELECT year(date) AS year, COUNT(*) AS bars,
           COUNT(DISTINCT security_id) AS securities
    FROM eod GROUP BY 1 ORDER BY 1
''').df()
by_year.head(30)

,year,bars,securities
0,2000,1896708,8356
1,2001,1760438,7827
2,2002,1695175,7303
3,2003,1673067,7385
4,2004,1737271,7518
5,2005,1760731,7648
6,2006,1789276,7822
7,2007,1847558,8146
8,2008,1888172,8101
9,2009,1861303,7993


## 2. Survivorship — the reason this dataset exists

The universe includes securities that stopped trading. A dataset of *current*
listings would quietly exclude every company that failed, which is the single
most damaging bias in equity backtesting.

In [4]:
con.execute('''
    SELECT s.is_delisted,
           COUNT(DISTINCT e.security_id) AS securities,
           MIN(e.date) AS first_bar, MAX(e.date) AS last_bar
    FROM eod e JOIN securities s USING (security_id)
    GROUP BY 1 ORDER BY 1
''').df()

,is_delisted,securities,first_bar,last_bar
0,False,12556,2000-01-03,2026-07-27
1,True,18957,2000-01-03,2026-07-24


In [5]:
# Securities whose price history ends well before the dataset does: these are
# the names a survivorship-biased dataset would be missing entirely.
con.execute('''
    SELECT s.api_ticker, s.name, s.venue,
           MIN(e.date) AS first_bar, MAX(e.date) AS last_bar, COUNT(*) AS bars
    FROM eod e JOIN securities s USING (security_id)
    WHERE s.is_delisted
    GROUP BY 1,2,3
    HAVING MAX(e.date) < DATE '2010-01-01'
    ORDER BY bars DESC
    LIMIT 10
''').df()

,api_ticker,name,venue,first_bar,last_bar,bars
0,EPEX.US,Edge Petroleum Corp,NASDAQ,2000-01-03,2009-12-31,2515
1,APO_old.US,American Community Properties Trust,NYSE,2000-01-03,2009-12-30,2514
2,BPURQ.US,Biopure Corp,NASDAQ,2000-01-03,2009-12-23,2510
3,RHDCQ.US,R H Donnelley Corp,NYSE,2000-01-03,2009-12-23,2510
4,NRGN.US,Neurogen Corp,NASDAQ,2000-01-03,2009-12-23,2510
5,NASMQ.US,North American Scientific Inc,NASDAQ,2000-01-03,2009-12-22,2509
6,MTSI1.US,Mts Medication Technologies Inc,NASDAQ,2000-01-03,2009-12-22,2509
7,ION_old.US,Ion Media Networks Inc,NYSE,2000-01-03,2009-12-22,2509
8,ADMGQ.US,Advanced Materials Group Inc,NASDAQ,2000-01-03,2009-12-22,2509
9,SMTL.US,Semitool Inc,NASDAQ,2000-01-03,2009-12-21,2508


## 3. Adjusted prices — the join that matters

`eod.close` is **raw**. To get a return series, multiply by `price_factor` from
`adjustments`, joined on `(security_id, date)`.

The factor is anchored so the most recent bar is exactly 1.0, and every earlier
bar carries the cumulative product of corporate actions after it.

In [6]:
px = con.execute('''
    SELECT e.date, e.close AS raw_close,
           e.close * a.price_factor AS adj_close,
           a.split_factor, a.price_factor, e.vendor_adjusted_close
    FROM eod e
    JOIN adj a USING (security_id, date)
    WHERE e.api_ticker = 'AAPL.US'
    ORDER BY e.date
''').df()

print(f"{len(px):,} bars")
print("\nAAPL around the 4-for-1 split on 2020-08-31:")
px[(px.date >= pd.Timestamp('2020-08-26')) & (px.date <= pd.Timestamp('2020-09-03'))]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

6,680 bars

AAPL around the 4-for-1 split on 2020-08-31:


,date,raw_close,adj_close,split_factor,price_factor,vendor_adjusted_close
5195,2020-08-26,506.09,126.5225,0.25,0.25,122.7235
5196,2020-08-27,500.04,125.0100,0.25,0.25,121.2564
5197,2020-08-28,499.23,124.8075,0.25,0.25,121.0600
5198,2020-08-31,129.04,129.0400,1.00,1.00,125.1654
5199,2020-09-01,134.18,134.1800,1.00,1.00,130.1511
5200,2020-09-02,131.40,131.4000,1.00,1.00,127.4545
5201,2020-09-03,120.88,120.8800,1.00,1.00,117.2504


Note the raw close drops ~4x across the split while the adjusted series is
continuous. A strategy computing returns from `raw_close` would see a fictional
-75% day.

In [7]:
raw_ret = px.set_index('date').raw_close.pct_change()
adj_ret = px.set_index('date').adj_close.pct_change()
split_day = pd.Timestamp('2020-08-31')
pd.DataFrame({
    'return from raw_close': [raw_ret.loc[split_day]],
    'return from adj_close': [adj_ret.loc[split_day]],
}, index=['2020-08-31'])

,return from raw_close,return from adj_close
2020-08-31,-0.741522,0.033912


## 4. Building a price matrix for the backtester

`BACKTEST.py` wants a date x ticker DataFrame of prices. Pivot the adjusted
series. Keep it to a small, liquid universe here — the full dataset is 59M bars
and pivoting all of it is not something to do casually.

In [8]:
UNIVERSE = ['AAPL.US','MSFT.US','JNJ.US','XOM.US','KO.US','IBM.US','GE.US','PG.US']

prices = con.execute(f'''
    SELECT e.date, e.api_ticker, e.close * a.price_factor AS px
    FROM eod e JOIN adj a USING (security_id, date)
    WHERE e.api_ticker IN ({",".join(f"'{t}'" for t in UNIVERSE)})
      AND e.date >= DATE '2010-01-01'
''').df().pivot(index='date', columns='api_ticker', values='px').sort_index()

print(prices.shape)
prices.tail(3)

(4165, 8)


api_ticker,AAPL.US,GE.US,IBM.US,JNJ.US,KO.US,MSFT.US,PG.US,XOM.US
date,,,,,,,,
2026-07-23,321.66,349.00,206.65,259.27,81.17,381.58,146.97,156.89
2026-07-24,333.02,353.73,214.19,263.40,82.25,381.70,147.41,156.94
2026-07-27,336.91,361.61,216.28,265.95,84.07,389.10,148.63,154.77


## 5. Feeding it to `BACKTEST.py`

Equal-weight, monthly rebalance, purely as an integration check that the dataset
plugs into the backtester. This is **not** a strategy.

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd() if Path.cwd().name == 'research' else ROOT / 'research'))
from BACKTEST import backtest

w = pd.DataFrame(1.0 / prices.shape[1], index=prices.index, columns=prices.columns)
rebal = prices.resample('ME').last().index

res = backtest(w, prices, signal_dates=list(rebal), transaction_cost=0.0005)
{k: (round(v, 4) if isinstance(v, float) else v)
 for k, v in res.items() if isinstance(v, (int, float))}

## 6. Caveats you must carry into any research

1. **These are price returns, not total returns.** Dividends have not been
   downloaded yet, so `price_factor` currently equals `split_factor`. Returns are
   systematically understated for dividend payers. Check
   `data/catalog/phase1_manifest.json` -> `adjustments.factor_meaning`.

2. **Join on `security_id`, never on ticker.** Symbols are recycled after
   delisting. `api_ticker` is in `eod` for convenience only.

3. **~13% of securities with corporate actions show events dated outside their
   own price history** — spliced or recycled series. Treat those with suspicion;
   see the `spliced / recycled tickers` check in `xcap phase1-qa`.

4. **Known vendor defects**, all small but present: ~0.04% of bars have
   non-positive prices, ~0.01% violate `low <= {open,close} <= high`. They are
   flagged rather than silently repaired, because dropping vs winsorising is a
   strategy decision.

5. **The universe is deliberately not filtered by liquidity or price.** It
   includes sub-penny microcaps. Apply your own eligibility screen.

In [10]:
import json
m = json.load(open(ROOT / 'data' / 'catalog' / 'phase1_manifest.json'))
print('start_date      :', m['start_date'])
print('skipped blocks  :', m.get('skipped', {}))
a = json.load(open(ROOT / 'data' / 'catalog' / 'progress.json'))
print('universe        :', f"{a['universe_size']:,} securities")

start_date      : 2000-01-01
skipped blocks  : {'dividends': '300/32,525 securities resolved - block incomplete, not built'}
universe        : 32,525 securities


---

# Two cross-sectional strategies

Both are long-only, equal-weighted **top 2%**, rebalanced monthly, on a
liquidity-screened universe.

| strategy | signal | reference |
|---|---|---|
| **hi52** | `price / 252-day high` — proximity to the 52-week high | George & Hwang (2004) |
| **mom12-1** | `P[t-21] / P[t-252] - 1` — 12-month return skipping the most recent month | Jegadeesh & Titman (1993) |

`mom12-1` skips the last month deliberately: short-horizon reversal contaminates
the raw 12-month return, and the skip is what separates momentum from it.

### Timing — why there is no look-ahead

Verified against the engine rather than assumed. In `_drift_core`, weights set at
index `i` earn `rets[i]`, and `rets = prices.pct_change()`, so `rets[i]` is the
return from `i-1` to `i`. With `signal_dates` on month-ends and `lag=0`, a signal
computed from data through month-end `t` is executed at `t+1` and earns the
return of month `t+1`. Every input to the signal precedes the return it earns.

In [11]:
import time
t0 = time.time()

# Daily panel: adjusted price for returns, split-adjusted dollar volume for
# liquidity. Dollar volume must pair the SPLIT-adjusted price with the vendor's
# split-adjusted volume -- mixing raw price with adjusted volume breaks across
# every split.
con.execute(f'''
CREATE OR REPLACE TABLE daily AS
SELECT e.security_id, e.date,
       e.close                               AS raw,
       e.close * a.price_factor              AS adj,
       e.close * a.split_factor * e.volume   AS dv
FROM read_parquet('{PARQUET}/eod/**/*.parquet') e
JOIN read_parquet('{PARQUET}/adjustments/**/*.parquet') a USING (security_id, date)
JOIN read_parquet('{PARQUET}/securities.parquet') s USING (security_id)
WHERE s.type = 'Common Stock' AND e.close > 0 AND a.price_factor > 0
''')

# Vendor data-quality guard. A missed reverse split shows up as an enormous
# one-day gain in the adjusted series: 6,750 monthly returns in this panel
# exceed +200% and only 251 have a split recorded. A momentum signal ranks on
# return, so without this guard the strategy preferentially buys the errors.
con.execute(f'''
CREATE OR REPLACE TABLE daily_qc AS
SELECT d.*,
       CASE WHEN d.adj / lag(d.adj) OVER (PARTITION BY d.security_id ORDER BY d.date) - 1 > 2.0
             AND sp.security_id IS NULL
            THEN 1 ELSE 0 END AS bad_jump
FROM daily d
LEFT JOIN (SELECT DISTINCT security_id, date
           FROM read_parquet('{PARQUET}/splits.parquet')) sp
  ON sp.security_id = d.security_id AND sp.date = d.date
''')

# Rolling signal inputs. Every window looks strictly backwards.
con.execute('''
CREATE OR REPLACE TABLE sig AS
SELECT security_id, date, raw, adj, dv,
       max(adj)  OVER w252 AS hi52,
       lag(adj,  21) OVER (PARTITION BY security_id ORDER BY date) AS p21,
       lag(adj, 252) OVER (PARTITION BY security_id ORDER BY date) AS p252,
       median(dv) OVER w63 AS dv63,
       count(*)   OVER w252 AS hist,
       sum(bad_jump) OVER w252 AS bad252
FROM daily_qc
WINDOW w252 AS (PARTITION BY security_id ORDER BY date ROWS BETWEEN 251 PRECEDING AND CURRENT ROW),
       w63  AS (PARTITION BY security_id ORDER BY date ROWS BETWEEN  62 PRECEDING AND CURRENT ROW)
''')

# A MARKET-WIDE month-end calendar. Taking each security's own last trading day
# of the month instead would give every name a different rebalance date, so the
# cross-section would collapse to one name per date and no portfolio could form.
con.execute('''
CREATE OR REPLACE TABLE cal AS
WITH cnt AS (SELECT date, COUNT(*) AS n FROM daily GROUP BY date),
     mth AS (SELECT date, n,
                    median(n) OVER (PARTITION BY date_trunc('month', date)) AS med
             FROM cnt)
SELECT max(date) AS me FROM mth
WHERE n >= 0.5 * med          -- sessions only, see below
GROUP BY date_trunc('month', date)
''')

# The breadth filter is load-bearing, not hygiene. A handful of securities carry
# stale bars on US market holidays, so a plain max(date) picks Good Friday
# 2002/2013/2018/2024 and Memorial Day 2004/2010/2021 as the month-end -- giving
# a "market-wide" cross-section of 1-5 names. Every other name is then NaN on
# that row AND on the next (pct_change needs both endpoints), so the whole
# portfolio is frozen at a 0% return for two months, seven times over.

# Month-end rows on the common calendar. A security that stopped trading before
# the month's last session simply has no row -- correct, it was not tradable.
con.execute('''
CREATE OR REPLACE TABLE month_end AS
SELECT security_id, date, raw, adj, dv63, bad252,
       adj / hi52     AS hi52_sig,
       p21 / p252 - 1 AS mom_sig
FROM sig
WHERE date IN (SELECT me FROM cal)
  AND hist = 252 AND p252 IS NOT NULL AND p252 > 0 AND hi52 > 0
''')

# --- strategy parameters -------------------------------------------------
MIN_PRICE  = 5.0      # on the AS-TRADED close, not the adjusted one -- see below
MIN_DOLLAR_VOL = 10e6 # median 63-day dollar volume; institutional liquidity floor
TOP_PCT    = 0.02     # trade the top 2% of the cross-section
# -------------------------------------------------------------------------

con.execute(f'''
CREATE OR REPLACE TABLE eligible AS
SELECT * FROM month_end
-- Screen on the AS-TRADED price, never the adjusted one. A shell that traded
-- at $0.0001 and later reverse-split 1:15000 carries an adjusted price of ~$150
-- in that era, so an adjusted-price filter waves it straight through and the
-- resulting returns are fiction.
WHERE raw >= {MIN_PRICE} AND dv63 >= {MIN_DOLLAR_VOL}
  AND bad252 = 0          -- no unexplained jump in the trailing year
''')

print("excluded by data-quality guard:",
      con.execute(f'''SELECT COUNT(*) FROM month_end
          WHERE raw >= {MIN_PRICE} AND dv63 >= {MIN_DOLLAR_VOL} AND bad252 > 0''').fetchone()[0])

print(f"built in {time.time()-t0:.1f}s")
con.execute('''SELECT COUNT(*) AS obs, COUNT(DISTINCT security_id) AS securities,
                      MIN(date) AS first, MAX(date) AS last FROM eligible''').df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

excluded by data-quality guard: 2961
built in 25.6s


,obs,securities,first,last
0,511973,6657,2000-12-29,2026-07-27


### The screen, and why it is not a bias

`price >= $5` and `median 63-day dollar volume >= $10M` are applied from trailing
data at each month-end, so a name enters and leaves the universe as its own
history dictates. Nothing is selected using future information.

The $10M floor matters more at a 2% selection than at a decile: concentrating
into ~40 names makes the portfolio far more sensitive to whether those names are
actually tradable in size. Without it, a 2% cut on a raw universe reliably
selects the most illiquid names, because thin stocks produce the most extreme
signal values in both directions.

The check that matters: **what share of eligible observations come from
securities that no longer exist?** If that number were near zero, the universe
would be survivors only and every result below would be worthless.

In [12]:
con.execute(f'''
    SELECT s.is_delisted,
           COUNT(*) AS monthly_obs,
           COUNT(DISTINCT e.security_id) AS securities,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_obs
    FROM eligible e JOIN securities s USING (security_id)
    GROUP BY 1 ORDER BY 1
''').df()

,is_delisted,monthly_obs,securities,pct_of_obs
0,False,329103,3025,64.3
1,True,182870,3632,35.7


In [13]:
# Breadth over time: how many names the decile is drawn from each month.
breadth = con.execute('''
    SELECT date, COUNT(*) AS n_eligible FROM eligible GROUP BY 1 ORDER BY 1
''').df()
breadth['date'] = pd.to_datetime(breadth['date'])
print(f"eligible names per month: min {breadth.n_eligible.min():,} | "
      f"median {int(breadth.n_eligible.median()):,} | max {breadth.n_eligible.max():,}")
breadth.set_index('date').n_eligible.resample('YE').mean().round(0).tail(10)

eligible names per month: min 1 | median 1,822 | max 2,338


date
2017-12-31    1976.0
2018-12-31    2057.0
2019-12-31    1932.0
2020-12-31    1995.0
2021-12-31    2030.0
2022-12-31    2086.0
2023-12-31    1939.0
2024-12-31    1995.0
2025-12-31    2140.0
2026-12-31    2280.0
Freq: YE-DEC, Name: n_eligible, dtype: float64

## Price and cost matrices

Columns are keyed on `security_id`, never on ticker — tickers are recycled after
delisting, and a recycled symbol would silently splice two companies into one
column.

The price matrix deliberately spans **all** month-ends for these securities, not
just eligible ones, so a position held into ineligibility still earns its return.

In [ ]:
# The backtest runs DAILY and rebalances monthly. A month-end price panel cannot
# see an intra-month delisting, stop or gap, and it loses two full months of return
# whenever a single month-end row is missing.
REBAL = [pd.Timestamp(d) for (d,) in con.execute('SELECT me FROM cal ORDER BY me').fetchall()]

daily_px = con.execute("""
    SELECT date, security_id, adj FROM daily_qc
    WHERE security_id IN (SELECT DISTINCT security_id FROM eligible)
""").df()
daily_px['date'] = pd.to_datetime(daily_px['date'])
prices = daily_px.pivot(index='date', columns='security_id', values='adj').sort_index()
del daily_px

# Execution-date inputs are only ever read on rebalance rows, so they stay month-end
# sized -- backtest() aligns them itself. raw is the AS-TRADED close: the per-share
# commission needs a real share count, which a back-adjusted price does not give.
me_rows = con.execute("""
    SELECT security_id, date, raw, dv63 FROM month_end
    WHERE security_id IN (SELECT DISTINCT security_id FROM eligible)
""").df()
me_rows['date'] = pd.to_datetime(me_rows['date'])
raw_px     = me_rows.pivot(index='date', columns='security_id', values='raw').reindex(columns=prices.columns)
dollar_vol = me_rows.pivot(index='date', columns='security_id', values='dv63').reindex(columns=prices.columns)

print(f"price matrix : {prices.shape[0]:,} sessions x {prices.shape[1]:,} securities "
      f"({prices.memory_usage(deep=True).sum()/1e9:.2f} GB)")
print(f"rebalances   : {len(REBAL)} month-ends")


In [ ]:
from BACKTEST import backtest, spread_costs, results_backtest, IBKR

CAPITAL     = 25_000_000     # order sizes, and therefore commission and impact, scale with this
# Square-root impact: cost += coef * sqrt(order notional / ADV). The standard form
# is c * daily_vol * sqrt(participation) with c ~ 0.5-1.0, so the coefficient is of
# the order of a daily standard deviation, not a round number. These names run
# ~2-3%/day, so 0.02 is the defensible middle; the result is very sensitive to it,
# which is the point -- this term is what bounds capacity.
IMPACT_COEF = 0.02

# spread_costs expects MONTHLY dollar volume; dv63 is a median DAILY figure, so it
# must be annualised to the month (~21 sessions). Feeding the daily number straight
# in demotes every name a tier or two and roughly triples the modelled spread.
tcost = spread_costs(dollar_vol * 21)
print("one-way spread by tier (fraction of traded notional):")
tcost.stack().describe(percentiles=[.1, .5, .9]).round(5)


## Portfolio construction — equal-weight top decile

In [ ]:
def top_pct_weights(signal_col: str, top_pct: float = TOP_PCT) -> pd.DataFrame:
    '''Equal-weight the top `top_pct` of the cross-section by `signal_col`.

    The threshold is computed within each date's own cross-section, so it adapts
    to breadth and never peeks across time.
    '''
    panel = con.execute(
        f"SELECT security_id, date, {signal_col} AS sig FROM eligible "
        f"WHERE {signal_col} IS NOT NULL"
    ).df()
    panel['date'] = pd.to_datetime(panel['date'])

    # Indexed on rebalance dates, not on the daily calendar -- backtest() only reads
    # this on signal dates, and a daily-sized weight frame is pure memory.
    w = pd.DataFrame(0.0, index=pd.DatetimeIndex(REBAL), columns=prices.columns)
    picks = []
    for d, g in panel.groupby('date'):
        if d not in w.index or len(g) < 100:      # need a meaningful cross-section
            continue
        # Take an exact count, not a quantile threshold. hi52_sig is capped at
        # 1.0, so a large mass of names sits exactly at its 52-week high; a
        # threshold sweeps in every tie and the "top 2%" silently becomes 15%.
        # Ties are broken deterministically by security_id for reproducibility.
        n = max(int(round(top_pct * len(g))), 10)
        sel = (g.sort_values(['sig', 'security_id'], ascending=[False, True])
                 .head(n).security_id.tolist())
        sel = [s for s in sel if s in w.columns]
        if sel:
            w.loc[d, sel] = 1.0 / len(sel)
            picks.append(len(sel))
    print(f"{signal_col}: {len(picks)} rebalances, "
          f"median {int(np.median(picks))} names held "
          f"(min {min(picks)}, max {max(picks)})")
    return w

w_hi52 = top_pct_weights('hi52_sig')
w_mom  = top_pct_weights('mom_sig')


In [ ]:
# freq=252 and lag=1: form the signal on the month-end close, trade the NEXT
# session's close. lag=0 would assume we trade the same close we ranked on.
common = dict(freq=252, lag=1, signal_dates=REBAL, transaction_cost=tcost,
              raw_prices=raw_px, dollar_volume=dollar_vol,
              capital=CAPITAL, impact_coef=IMPACT_COEF, **IBKR)
res = {
    'hi52 (top 2%)':    backtest(w_hi52, prices, **common),
    'mom12-1 (top 2%)': backtest(w_mom,  prices, **common),
}

pd.DataFrame({
    k: {m: v[m] for m in ('total_return','ann_return','ann_vol','sharpe','sortino_ratio',
                          'max_drawdown','win_rate','ann_turnover',
                          'ann_cost_drag','ann_commission_drag','ann_impact_drag')}
    for k, v in res.items()
}).T.round(4)


In [ ]:
out = results_backtest(res, title='hi52 vs mom12-1 - equal-weight top 2%, daily, monthly rebalance')
out['summary_df']


## How much does the delisting assumption matter?

The engine cannot hold a name that has no price on the execution date, so a
security that stops trading between signal and execution is dropped from the
portfolio. Its final, usually adverse, move is never earned. That biases these
results **upward**.

The size of the effect is measurable: count signal-date selections that had no
price at the following month-end.

In [ ]:
# Per-year breakdown. Return is CHAINED from daily returns inside each year --
# equity[last]/equity[first] would drop every January.
res['hi52 (top 2%)']['yearly'][
    ['Periods','Return','Vol','Sharpe','MaxDD','WinRate','Turnover','Cost','Impact','Trades']
].round(4)


In [ ]:
# Trade blotter. `cost_frac` is the all-in one-way cost charged on that order
# (spread + commission + impact); `cost` is that in dollars.
blotter = res['hi52 (top 2%)']['trades']
print(f"{len(blotter):,} trades, {blotter.date.nunique()} rebalances, "
      f"${blotter.cost.sum():,.0f} total cost on ${blotter.notional.sum():,.0f} traded")
blotter.head(12)


In [ ]:
# A name that delists mid-month leaves the panel, so the engine holds it flat and
# closes it at the next rebalance with no execution price. Those rows are the
# positions whose real exit return is unknown -- the remaining known bias.
nan_px = blotter[blotter.price.isna()]
print(f"{len(nan_px):,} of {len(blotter):,} trades close at an unknown price "
      f"({100*len(nan_px)/max(len(blotter),1):.2f}%)")
nan_px.head()


## Reading these numbers honestly

**What is sound.** The universe is survivorship-free: delisted securities are
present and supply a large share of eligible observations. Signals use only
trailing data, the eligibility screen is applied point-in-time, costs are
liquidity-tiered rather than a flat optimistic constant, and the signal-to-return
timing was verified against the engine rather than assumed.

**What is not, yet.**

1. **Price returns, not total returns.** Dividends are not downloaded, so
   `price_factor == split_factor`. Both strategies are understated, and *not
   equally* — whichever tilts toward higher-yielding names is penalised more, so
   the comparison between them is affected, not just the levels.

2. **Delisting returns are absent.** Quantified above. Positions that vanish
   between signal and execution are silently skipped instead of taking their loss.
   The standard correction applies roughly -30% to involuntary delistings, which
   needs delisting-reason data this vendor does not provide.

3. **No corporate-action triage.** ~13% of securities with corporate actions have
   events dated outside their own price history (spliced or recycled tickers).
   They are still in this universe.

4. **Long-only, no benchmark.** Neither result is separated from plain market
   beta. A long-short decile spread or a market-excess return would be the
   meaningful test.

5. **Concentration risk is now material.** A 2% cut holds ~40 names, so
   idiosyncratic variance dominates and the metrics are far noisier than the
   decile version. Sharpe differences of a few tenths between the two strategies
   are not distinguishable at this breadth.

6. **One parameter set, one sample.** 2% cut, $5/$10M screen, monthly rebalance,
   2000-2026, all chosen up front and not varied. Treat these as a pipeline
   demonstration, not evidence about either factor.